In [ ]:
import numpy as np
import pandas as pd

import copy
import gc
import json
import os
from pathlib import Path

from scipy.sparse import issparse
import matplotlib.pyplot as plt
from anndata import AnnData
import scanpy as sc

import torch
from torchtext.vocab import Vocab
from torchtext._torchtext import (
    Vocab as VocabPybind,
)

import scgpt as scg
from scgpt.model import MultiOmicTransformerModel
from scgpt.scbank import DataBank
from scgpt.tokenizer import GeneVocab, get_default_gene_vocab
from scgpt.utils import set_seed
from scgpt.preprocess import Preprocessor

In [ ]:
class config:
    # static configs
    model_path = '../scGPT_human'
    pad_token = "<pad>"
    mask_value = -1,
    pad_value = -2,
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    seed = 42

    # sampling/preprocessing
    n_hvg = 1200 # don't do hvg in the pretraining run as you would lose so much information
    n_hvp = 4000 # still do hvp since there's too many peaks to attend to (too many additoinal vocabs), but slightly higher count should be allowed
    n_bins = 51
    input_layer_key = "X_binned"

    # architecture
    layer_size = 512
    nhead = 8 # # if load model, batch_size, layer_size, nlayers, nhead will be ignored
    nlayers = 4 # for developing
    freeze = False # freeze

    # objectives
    GEP=True # Gene expression modelling
    GEPC=True # Gene expression modelling for cell objective
    CLS=False
    ESC=False
    DAR = False # DAR objective weight for batch correction
    DSBN = False  # Domain-spec batchnorm,

    # training hyperparams
    mask_ratio=0.4 # Default mask ratio
    explicit_zero_prob = False  # whether explicit bernoulli for zeros
    ecs_thres=0  # Elastic cell similarity objective, 0.0 to 1.0, 0.0 to disable
    dab_weight=1.0

    # training settings
    use_batch_labels = False
    use_mod = True # specify modality
    pre_norm=False # Default setting
    include_zero_gene = False

    # sequence handling
    per_seq_batch_sample = False
    max_seq_len = 1201 # # Default n_hvg+1

    # optimization
    epochs=1 # ?
    lr=1e-4 # Default learning rate 
    batch_size=16 # Default batch size
    dropout=0.2 # Default dropout rate
    schedule_ratio=0.95  # Default rate for learning rate decay
    fast_transformer=True # Default setting
    amp=True  # Default setting: Automatic Mixed Precision

    # logging
    save_eval_interval=5 # Default model evaluation interval
    log_interval=100 # Default log interval

    


# settings for input and preprocessing
special_tokens = [config.pad_token, "<cls>", "<eoc>"]
set_seed(config.seed)
logger = scg.logger